# Phase 8: Zerodha Data Fetcher for Bear Market Test (Subset 10)

Downloads historical OHLCV data for F&O eligible stocks (Apr 2020 - Nov 2022).

**Prerequisites:**
- Run `phase8_get_fno_universe.py` first to create the F&O universe list
- Zerodha account with API access

**Output:**
- `data/NIFTY200_Subset10/raw/{SYMBOL}.csv` (one file per stock)
- `data/NIFTY200_Subset10/data_quality_report.txt`

## 1. Setup and Imports

In [1]:
import os
import sys
import time
import logging
import json
from datetime import datetime, timedelta, date
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
from kiteconnect import KiteConnect

# Change to project root directory
os.chdir('/home/ubuntu/rajnish/Multitask-Stockformer')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/ubuntu/rajnish/Multitask-Stockformer


## 2. Configuration

**Update date range and paths as needed:**

In [2]:
# Zerodha API credentials (same as Phase 2)
API_KEY = "a3vlmmcvyt40udoq"
API_SECRET = "xin86nvnojty5996zzexbu7chc040zy0"

# Date range for Subset 10 (Apr 2020 - Nov 2022)
FROM_DATE = "2020-04-01"
TO_DATE = "2022-11-30"

# Paths
OUTPUT_DIR = "./data/NIFTY200_Subset10"
RAW_DATA_DIR = os.path.join(OUTPUT_DIR, "raw")
UNIVERSE_FILE = os.path.join(OUTPUT_DIR, "fno_universe_2020_04.txt")
TOKEN_FILE = os.path.join(OUTPUT_DIR, "zerodha_token.json")
LOG_FILE = os.path.join(OUTPUT_DIR, "zerodha_fetcher.log")

# Data quality thresholds
REQUIRED_COVERAGE = 0.80  # Require 80% coverage of trading days

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RAW_DATA_DIR, exist_ok=True)

print(f"API Key: ✓ Configured")
print(f"Date Range: {FROM_DATE} to {TO_DATE}")
print(f"Output Directory: {OUTPUT_DIR}")

API Key: ✓ Configured
Date Range: 2020-04-01 to 2022-11-30
Output Directory: ./data/NIFTY200_Subset10


## 3. Configure Logging

In [3]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info("Logger configured successfully")

2026-01-11 11:37:56,050 - INFO - Logger configured successfully


## 4. Token Management Functions

In [4]:
def save_token(access_token: str) -> dict:
    """Save access token with timestamp to file"""
    token_data = {
        'access_token': access_token,
        'timestamp': datetime.now().isoformat(),
        'date': datetime.now().date().isoformat()
    }
    
    with open(TOKEN_FILE, 'w') as f:
        json.dump(token_data, f, indent=2)
    
    logger.info(f"Token saved to {TOKEN_FILE}")
    return token_data

def load_token() -> Optional[str]:
    """Load access token if valid (same day)"""
    if not os.path.exists(TOKEN_FILE):
        logger.info("No saved token found")
        return None
    
    try:
        with open(TOKEN_FILE, 'r') as f:
            token_data = json.load(f)
        
        # Check if token is from today
        saved_date = date.fromisoformat(token_data['date'])
        today = datetime.now().date()
        
        if saved_date == today:
            logger.info(f"✓ Found valid token from {token_data['timestamp']}")
            return token_data['access_token']
        else:
            logger.info(f"Token expired (from {saved_date}), need fresh login")
            return None
    
    except Exception as e:
        logger.error(f"Error loading token: {e}")
        return None

print("✓ Token management functions defined")

✓ Token management functions defined


## 5. Initialize Zerodha API Connection

**If token is expired, this cell will show the login URL. Follow the instructions to get a new token.**

In [5]:
# Initialize KiteConnect
kite = KiteConnect(api_key=API_KEY)

# Try to load existing token
access_token = load_token()

if access_token:
    # Use saved token
    kite.set_access_token(access_token)
    print("✅ Using saved session - no login required!")
else:
    # Manual login required
    print("\n" + "="*80)
    print("MANUAL LOGIN REQUIRED (One-time per day)")
    print("="*80)
    print(f"\n1. Open this URL in your browser:\n   {kite.login_url()}")
    print("\n2. After login, you'll be redirected to: http://127.0.0.1/?request_token=...")
    print("\n3. Copy the 'request_token' value and paste it in the next cell")

2026-01-11 11:38:10,817 - INFO - No saved token found



MANUAL LOGIN REQUIRED (One-time per day)

1. Open this URL in your browser:
   https://kite.zerodha.com/connect/login?api_key=a3vlmmcvyt40udoq&v=3

2. After login, you'll be redirected to: http://127.0.0.1/?request_token=...

3. Copy the 'request_token' value and paste it in the next cell


## 6. Complete Login (if needed)

**Only run this cell if the previous cell showed "MANUAL LOGIN REQUIRED"**

Paste your request_token below:

In [6]:
# ONLY RUN THIS IF LOGIN REQUIRED
# Paste your request_token here:
request_token = "erR3eiW4oWxYCJA95QLFMn8KFbKqytLb"  # <-- Paste token here between the quotes

if request_token:
    try:
        data = kite.generate_session(request_token, api_secret=API_SECRET)
        access_token = data["access_token"]
        kite.set_access_token(access_token)
        save_token(access_token)
        print("✅ Login Successful! Token saved for today.")
    except Exception as e:
        print(f"❌ Login Failed: {e}")
else:
    print("ℹ️ No request_token provided. Skip this cell if already logged in.")

2026-01-11 11:39:29,847 - INFO - Token saved to ./data/NIFTY200_Subset10/zerodha_token.json


✅ Login Successful! Token saved for today.


## 7. Test Connection

In [7]:
# Test connection
try:
    profile = kite.profile()
    print(f"\n✅ Connected to Zerodha API")
    print(f"User: {profile['user_name']}")
    print(f"Broker: {profile['broker']}")
    print(f"Email: {profile['email']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\nPlease complete the login in the previous cell first.")


✅ Connected to Zerodha API
User: Rajnish Ahuja
Broker: ZERODHA
Email: rajnish.ahuja82@gmail.com


## 8. Load F&O Universe

In [12]:
# Check if universe file exists
if not os.path.exists(UNIVERSE_FILE):
    print(f"❌ Error: Universe file not found: {UNIVERSE_FILE}")
    print("Please run phase8_get_fno_universe.py first!")
else:
    # Load F&O universe
    with open(UNIVERSE_FILE, 'r') as f:
        symbols = [line.strip() for line in f if line.strip()]
    
    print(f"✓ Loaded {len(symbols)} stocks from F&O universe")
    print(f"\nFirst 20: {symbols[:20]}")
    print(f"\nLast 10: {symbols[-10:]}")

✓ Loaded 144 stocks from F&O universe

First 20: ['ACC', 'ADANIENT', 'ADANIPORTS', 'ADANIPOWER', 'AMARAJABAT', 'AMBUJACEM', 'APOLLOHOSP', 'APOLLOTYRE', 'ASHOKLEY', 'ASIANPAINT', 'AUROPHARMA', 'AXISBANK', 'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BALKRISIND', 'BANDHANBNK', 'BANKBARODA', 'BATAINDIA', 'BEL']

Last 10: ['TVSMOTOR', 'UBL', 'UJJIVAN', 'ULTRACEMCO', 'UPL', 'VEDL', 'VOLTAS', 'WIPRO', 'YESBANK', 'ZEEL']


## 9. Fetch Instruments & Apply Symbol Mapping

In [26]:
# Step 1: Fetch NSE instruments list
print("Fetching NSE instruments list...")
instruments = kite.instruments("NSE")
instrument_map = {i['tradingsymbol']: i['instrument_token'] for i in instruments}
print(f"✓ Fetched {len(instruments)} NSE instruments")

# Step 2: Force reload symbol mapping (in case it was updated)
import importlib
import phase8_symbol_mapping
importlib.reload(phase8_symbol_mapping)
from phase8_symbol_mapping import apply_symbol_mapping, SYMBOL_MAPPING, SKIP_SYMBOLS

# Step 3: Apply mapping to get final available_symbols
available_symbols, mapped_symbols, skipped_symbols = apply_symbol_mapping(symbols, instrument_map)

# Summary
print(f"\n{'='*60}")
print(f"SYMBOL MAPPING SUMMARY")
print(f"{'='*60}")
print(f"Original F&O universe (Apr 2020): {len(symbols)} stocks")
print(f"Direct matches on NSE: {len(available_symbols) - len(mapped_symbols)}")
print(f"Mapped (renamed stocks): {len(mapped_symbols)}")
print(f"Skipped (no successor): {len(skipped_symbols)}")
print(f"\n✓ FINAL: {len(available_symbols)} stocks available for download")

if mapped_symbols:
    print(f"\nMapped symbols:")
    for m in mapped_symbols:
        print(f"    {m}")

if skipped_symbols:
    print(f"\n⚠ Skipped: {skipped_symbols}")

Fetching NSE instruments list...


✓ Fetched 9128 NSE instruments

SYMBOL MAPPING SUMMARY
Original F&O universe (Apr 2020): 144 stocks
Direct matches on NSE: 126
Mapped (renamed stocks): 18
Skipped (no successor): 0

✓ FINAL: 144 stocks available for download

Mapped symbols:
    AMARAJABAT -> ARE&M
    CADILAHC -> ZYDUSLIFE
    CENTURYTEX -> ABREL
    EQUITAS -> EQUITASBNK
    GMRINFRA -> GMRAIRPORT
    HDFC -> HDFCBANK
    IBULHSGFIN -> IBULLSLTD
    INFRATEL -> INDUSTOWER
    L&TFH -> LTF
    MCDOWELL-N -> UNITDSPR
    MINDTREE -> LTIM
    MOTHERSUMI -> MOTHERSON
    NIITTECH -> COFORGE
    PEL -> POONAWALLA
    PVR -> PVRINOX
    SRTRANSFIN -> SHRIRAMFIN
    TATAMOTORS -> TMPV
    UJJIVAN -> UJJIVANSFB


## 10. Define Data Fetching Functions

In [27]:
def fetch_historical_data(symbol: str, instrument_token: int,
                         from_date: str, to_date: str, 
                         max_retries: int = 5) -> Optional[pd.DataFrame]:
    """Fetch historical OHLCV data with exponential backoff retry"""
    from_dt = datetime.strptime(from_date, '%Y-%m-%d')
    to_dt = datetime.strptime(to_date, '%Y-%m-%d')
    
    for attempt in range(max_retries):
        try:
            historical_data = kite.historical_data(
                instrument_token=instrument_token,
                from_date=from_dt,
                to_date=to_dt,
                interval='day'
            )
            
            if not historical_data:
                logger.warning(f"No data returned for {symbol}")
                return None
            
            # Convert to DataFrame
            df = pd.DataFrame(historical_data)
            df = df[['date', 'open', 'high', 'low', 'close', 'volume']]
            df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
            df['Date'] = pd.to_datetime(df['Date']).dt.date
            
            logger.info(f"✓ Fetched {len(df)} records for {symbol}")
            return df
            
        except Exception as e:
            wait_time = 1 * (2 ** attempt)
            logger.warning(f"Attempt {attempt + 1}/{max_retries} failed for {symbol}: {e}")
            
            if attempt < max_retries - 1:
                logger.info(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                logger.error(f"✗ Failed after {max_retries} attempts: {symbol}")
                return None
    
    return None

def validate_data(df: pd.DataFrame, symbol: str) -> Dict:
    """Validate data completeness and quality"""
    from_dt = datetime.strptime(FROM_DATE, '%Y-%m-%d').date()
    to_dt = datetime.strptime(TO_DATE, '%Y-%m-%d').date()
    
    validation = {
        'symbol': symbol,
        'total_records': len(df),
        'date_range': f"{df['Date'].min()} to {df['Date'].max()}",
        'coverage': 0.0,
        'zero_volume_days': 0,
        'large_gaps': 0,
        'data_quality': 'PASS'
    }
    
    # Calculate expected trading days (rough estimate: 252 days/year)
    total_days = (to_dt - from_dt).days
    expected_trading_days = int(total_days * 252 / 365)
    actual_days = len(df)
    
    validation['coverage'] = actual_days / expected_trading_days if expected_trading_days > 0 else 0
    
    if validation['coverage'] < REQUIRED_COVERAGE:
        validation['data_quality'] = 'FAIL'
        logger.warning(f"{symbol}: Insufficient coverage {validation['coverage']:.1%}")
    
    # Check for zero volume days
    zero_vol = (df['Volume'] == 0).sum()
    validation['zero_volume_days'] = int(zero_vol)
    if zero_vol > len(df) * 0.1:
        validation['data_quality'] = 'WARN'
    
    # Check for large price gaps
    df_sorted = df.sort_values('Date')
    pct_change = df_sorted['Close'].pct_change().abs()
    large_gaps = (pct_change > 0.40).sum()
    validation['large_gaps'] = int(large_gaps)
    if large_gaps > 0:
        validation['data_quality'] = 'WARN'
    
    return validation

print("✓ Data fetching functions defined")

✓ Data fetching functions defined


In [28]:
# TEST: Download ALL stocks but only 1 day to verify
TEST_MODE = True
TEST_DATE = "2022-01-03"

if TEST_MODE:
    print(f"TEST MODE: {len(available_symbols)} stocks for {TEST_DATE}")
    success, failed = 0, []
    
    for idx, symbol in enumerate(available_symbols, 1):
        token = instrument_map.get(symbol)
        if token:
            df = fetch_historical_data(symbol, token, TEST_DATE, TEST_DATE)
            if df is not None and len(df) > 0:
                success += 1
                if idx <= 3 or idx % 50 == 0:
                    print(f"[{idx}] ✓ {symbol}")
            else:
                failed.append(symbol)
        else:
            failed.append(symbol)
        time.sleep(0.3)
    
    print(f"\n✓ {success}/{len(available_symbols)} successful")
    if failed: print(f"Failed: {failed}")

TEST MODE: 144 stocks for 2022-01-03


2026-01-11 12:23:27,698 - INFO - ✓ Fetched 1 records for ACC


[1] ✓ ACC


2026-01-11 12:23:28,481 - INFO - ✓ Fetched 1 records for ADANIENT


[2] ✓ ADANIENT


...

[50] ✓ GMRAIRPORT


2026-01-11 12:23:57,420 - INFO - ✓ Fetched 1 records for GODREJCP
2026-01-11 12:23:57,989 - INFO - ✓ Fetched 1 records for GODREJPROP
2026-01-11 12:23:58,559 - INFO - ✓ Fetched 1 records for GRASIM
2026-01-11 12:23:59,122 - INFO - ✓ Fetched 1 records for HAVELLS
2026-01-11 12:23:59,688 - INFO - ✓ Fetched 1 records for HCLTECH
2026-01-11 12:24:00,256 - INFO - ✓ Fetched 1 records for HDFCBANK
2026-01-11 12:24:00,822 - INFO - ✓ Fetched 1 records for HDFCBANK
2026-01-11 12:24:01,389 - INFO - ✓ Fetched 1 records for HDFCLIFE
2026-01-11 12:24:01,954 - INFO - ✓ Fetched 1 records for HEROMOTOCO
2026-01-11 12:24:02,521 - INFO - ✓ Fetched 1 records for HINDALCO
2026-01-11 12:24:03,128 - INFO - ✓ Fetched 1 records for HINDPETRO
2026-01-11 12:24:03,699 - INFO - ✓ Fetched 1 records for HINDUNILVR
2026-01-11 12:24:04,303 - INFO - ✓ Fetched 1 records for IBULLSLTD
2026-01-11 12:24:04,869 - INFO - ✓ Fetched 1 records for ICICIBANK
2026-01-11 12:24:05,435 - INFO - ✓ Fetched 1 records for ICICIPRULI
202

[100] ✓ COFORGE


2026-01-11 12:24:25,971 - INFO - ✓ Fetched 1 records for NMDC
2026-01-11 12:24:26,536 - INFO - ✓ Fetched 1 records for NTPC
2026-01-11 12:24:27,112 - INFO - ✓ Fetched 1 records for OIL
2026-01-11 12:24:27,717 - INFO - ✓ Fetched 1 records for ONGC
2026-01-11 12:24:28,286 - INFO - ✓ Fetched 1 records for PAGEIND
2026-01-11 12:24:29,070 - INFO - ✓ Fetched 1 records for POONAWALLA
2026-01-11 12:24:29,639 - INFO - ✓ Fetched 1 records for PETRONET
2026-01-11 12:24:30,209 - INFO - ✓ Fetched 1 records for PFC
2026-01-11 12:24:30,776 - INFO - ✓ Fetched 1 records for PIDILITIND
2026-01-11 12:24:31,380 - INFO - ✓ Fetched 1 records for PNB
2026-01-11 12:24:31,986 - INFO - ✓ Fetched 1 records for POWERGRID
2026-01-11 12:24:32,552 - INFO - ✓ Fetched 1 records for PVRINOX
2026-01-11 12:24:33,158 - INFO - ✓ Fetched 1 records for RAMCOCEM
2026-01-11 12:24:33,764 - INFO - ✓ Fetched 1 records for RBLBANK
2026-01-11 12:24:34,327 - INFO - ✓ Fetched 1 records for RECLTD
2026-01-11 12:24:34,896 - INFO - ✓ Fe


✓ 144/144 successful


## 11. Download All Data

**This will take ~1-2 hours for 144 stocks due to API rate limits.**

In [29]:
# Initialize tracking
validation_results = []
successful = 0
failed = []

print("="*80)
print(f"Starting data download for {len(available_symbols)} stocks")
print(f"Date range: {FROM_DATE} to {TO_DATE}")
print("="*80 + "\n")

# Download data for each symbol
for idx, symbol in enumerate(available_symbols, 1):
    print(f"\n[{idx}/{len(available_symbols)}] Processing {symbol}...")
    
    instrument_token = instrument_map[symbol]
    
    # Fetch data
    df = fetch_historical_data(symbol, instrument_token, FROM_DATE, TO_DATE)
    
    if df is not None and len(df) > 0:
        # Validate data
        validation = validate_data(df, symbol)
        validation_results.append(validation)
        
        # Save to CSV only if passes quality check
        if validation['data_quality'] != 'FAIL':
            output_file = os.path.join(RAW_DATA_DIR, f"{symbol}.csv")
            df.to_csv(output_file, index=False)
            print(f"✓ Saved {len(df)} records to {output_file}")
            successful += 1
        else:
            print(f"⚠ Skipped {symbol} due to insufficient data coverage")
            failed.append(symbol)
        
        # Rate limiting (Zerodha API limit)
        time.sleep(0.5)
    else:
        failed.append(symbol)
        print(f"✗ Failed to download {symbol}")

print("\n" + "="*80)
print("DOWNLOAD COMPLETE")
print("="*80)
print(f"Successful: {successful}/{len(available_symbols)}")
print(f"Failed: {len(failed)}")
if failed:
    print(f"\nFailed symbols: {', '.join(failed[:20])}{'...' if len(failed) > 20 else ''}")

Starting data download for 144 stocks
Date range: 2020-04-01 to 2022-11-30


[1/144] Processing ACC...


2026-01-11 12:45:11,958 - INFO - ✓ Fetched 662 records for ACC


✓ Saved 662 records to ./data/NIFTY200_Subset10/raw/ACC.csv

[2/144] Processing ADANIENT...


2026-01-11 12:45:12,844 - INFO - ✓ Fetched 662 records for ADANIENT


✓ Saved 662 records to ./data/NIFTY200_Subset10/raw/ADANIENT.csv

[3/144] Processing ADANIPORTS...


...

✓ Saved 662 records to ./data/NIFTY200_Subset10/raw/WIPRO.csv

[143/144] Processing YESBANK...


2026-01-11 12:47:14,373 - INFO - ✓ Fetched 662 records for YESBANK


✓ Saved 662 records to ./data/NIFTY200_Subset10/raw/YESBANK.csv

[144/144] Processing ZEEL...


2026-01-11 12:47:15,228 - INFO - ✓ Fetched 662 records for ZEEL


✓ Saved 662 records to ./data/NIFTY200_Subset10/raw/ZEEL.csv

DOWNLOAD COMPLETE
Successful: 143/144
Failed: 1

Failed symbols: EQUITASBNK


## 12. Generate Quality Report

In [30]:
# Save quality report
report_file = os.path.join(OUTPUT_DIR, "data_quality_report.txt")
with open(report_file, 'w') as f:
    f.write("Phase 8: Data Quality Report\n")
    f.write("="*60 + "\n\n")
    f.write(f"Date Range: {FROM_DATE} to {TO_DATE}\n")
    f.write(f"Universe: F&O stocks as of Apr 2020\n")
    f.write(f"Total stocks in universe: {len(symbols)}\n")
    f.write(f"Successfully downloaded: {successful}\n")
    f.write(f"Failed/Skipped: {len(failed)}\n\n")
    
    f.write("Validation Summary:\n")
    f.write("-"*60 + "\n")
    
    # Count by quality status
    pass_count = sum(1 for v in validation_results if v['data_quality'] == 'PASS')
    warn_count = sum(1 for v in validation_results if v['data_quality'] == 'WARN')
    fail_count = sum(1 for v in validation_results if v['data_quality'] == 'FAIL')
    
    f.write(f"PASS: {pass_count}\n")
    f.write(f"WARN: {warn_count}\n")
    f.write(f"FAIL: {fail_count}\n\n")
    
    if failed:
        f.write(f"Failed symbols:\n{', '.join(failed)}\n")

print(f"\n✓ Quality report saved to: {report_file}")
print(f"\nSummary:")
print(f"  PASS: {pass_count}")
print(f"  WARN: {warn_count}")
print(f"  FAIL: {fail_count}")


✓ Quality report saved to: ./data/NIFTY200_Subset10/data_quality_report.txt

Summary:
  PASS: 142
  WARN: 1
  FAIL: 1
